# femtosecond PINN — Colab runner

Thin launcher only (GUIDE.md §4.6). No model or physics code lives here.

1. Runtime → Change runtime type → **GPU**
2. Add a Colab **Secret** named `GITHUB_TOKEN` (fine-grained PAT with read access to the private repo)
3. Run all cells

In [ ]:
# 1. GPU + Drive
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone / pull the repo (token from Colab Secrets, never pasted here)
import os, subprocess
from google.colab import userdata

REPO = 'leejoohyunn/pinn-femtosecond-laser-ablation'
WORK = '/content/femtosecond'
token = userdata.get('GITHUB_TOKEN')
url = f'https://{token}@github.com/{REPO}.git'

if os.path.isdir(WORK):
    subprocess.run(['git', '-C', WORK, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', url, WORK], check=True)
    subprocess.run(['git', '-C', WORK, 'remote', 'set-url', 'origin', f'https://github.com/{REPO}.git'], check=True)  # drop token from .git/config
%cd {WORK}
!git log --oneline -1

In [ ]:
# 3. Install (torch is preinstalled on Colab; deepxde version pinned in requirements.txt)
%pip install -q -e . -r requirements.txt
%env DDE_BACKEND=pytorch

In [ ]:
# 4. outputs/ → Google Drive so results survive disconnects
import os
DRIVE_OUT = '/content/drive/MyDrive/femtosecond/outputs'
os.makedirs(DRIVE_OUT, exist_ok=True)
if os.path.islink('outputs'):
    os.unlink('outputs')
elif os.path.isdir('outputs'):
    import shutil; shutil.rmtree('outputs')
os.symlink(DRIVE_OUT, 'outputs')
!ls -la outputs

In [ ]:
# 5. Phase 0 smoke check (DoD: glass n_cr 1.83e27 m⁻³, device = cuda)
!python scripts/check_env.py
!pytest -q

## Launch commands (Phase 4, GUIDE §5 Phase 4 / §4.6)

Same scripts as on the Mac. The FDM references are regenerated here (deterministic, 0.05 s) so nothing is copied from the Mac.
Run the next cell once per session; it skips finished steps when the output folders already exist on Drive.
If the session drops during `full`, re-run the cell: it resumes inside the stage schedule (I-24).

In [ ]:
# 6. Phase 4: references → GPU smoke → full → evaluate (idempotent; re-run the same cell after a disconnect)
import glob, json, os, subprocess

def latest(pattern):
    hits = sorted(glob.glob(pattern))
    return hits[-1] if hits else None

def done_iters(run):
    """iterations_total of a finished train() call (metrics.json is written only at the end)."""
    m = None if run is None else os.path.join(run, 'metrics.json')
    return json.load(open(m))['iterations_total'] if m and os.path.exists(m) else 0

def run(cmd):
    """Stream the child's output into the cell (subprocess.run would swallow it) and fail loudly."""
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'exit code {p.returncode}: {cmd}')

if latest('outputs/fdm/glass_ref_rk4_2*') is None:          # deterministic, 0.05 s: never copied from the Mac
    run('python scripts/run_fdm.py --material glass --name glass_ref_rk4')
    run('python scripts/run_fdm.py --material glass --ionization photo_only --name glass_ref_rk4_photo')
fdm = latest('outputs/fdm/glass_ref_rk4_2*')

smoke = latest('outputs/forward/colab_smoke_gs_*')          # GPU sanity check at the calibrated physics (~minutes)
if done_iters(smoke) < 6000:
    run('python scripts/train_forward.py --config configs/forward_glass.yaml --profile smoke_gs --name colab_smoke_gs --set dtype=float32')
    smoke = latest('outputs/forward/colab_smoke_gs_*')
    run(f'python scripts/evaluate.py --run {smoke} --fdm {fdm}')

full = latest('outputs/forward/glass_full_*')
if full is None:
    run('python scripts/train_forward.py --config configs/forward_glass.yaml --profile full --name glass_full')
elif done_iters(full) < 50000:
    run(f'python scripts/train_forward.py --resume {full}')     # continues inside the stage schedule (I-24)
full = latest('outputs/forward/glass_full_*')
run(f'python scripts/evaluate.py --run {full} --fdm {fdm}')
